# Vision Transformers from Scratch

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zhubarb/sesen_ai_ml_tutorials/blob/main/notebooks/deep-learning/vision_transformers_from_scratch.ipynb)

Build a Vision Transformer (ViT) in PyTorch, train it on CIFAR-10, and compare it against a small CNN to see the inductive-bias gap first-hand. Companion to the [blog post](https://sesen.ai/blog/vision-transformers-from-scratch).

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms

device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print(device)

## 1. The Vision Transformer

### Patch embedding
An image becomes a sequence of patch tokens. A strided convolution with `kernel = stride = patch_size` cuts and linearly embeds every patch in one operation.

In [ ]:
class PatchEmbedding(nn.Module):
    def __init__(self, img_size=32, patch=4, in_chans=3, dim=192):
        super().__init__()
        self.n_patches = (img_size // patch) ** 2
        self.proj = nn.Conv2d(in_chans, dim, kernel_size=patch, stride=patch)

    def forward(self, x):
        x = self.proj(x)                      # (B, dim, 8, 8)
        return x.flatten(2).transpose(1, 2)   # (B, 64, dim)

tokens = PatchEmbedding()(torch.randn(1, 3, 32, 32))
print(tokens.shape)  # (1, 64, 192)

### Multi-head self-attention
Every token attends to every other token: `softmax(Q Kᵀ / √d) V`.

In [ ]:
class MultiHeadSelfAttention(nn.Module):
    def __init__(self, dim, n_heads):
        super().__init__()
        self.n_heads, self.head_dim = n_heads, dim // n_heads
        self.scale = self.head_dim ** -0.5
        self.qkv = nn.Linear(dim, dim * 3)
        self.proj = nn.Linear(dim, dim)

    def forward(self, x):
        B, N, C = x.shape
        qkv = self.qkv(x).reshape(B, N, 3, self.n_heads, self.head_dim).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        attn = (q @ k.transpose(-2, -1)) * self.scale
        attn = attn.softmax(dim=-1)
        out = (attn @ v).transpose(1, 2).reshape(B, N, C)
        return self.proj(out)

### Encoder block and the full model
Pre-norm attention + MLP, each with a residual. A learned class token summarises the image; learned positional embeddings tell the model where each patch sits.

In [ ]:
class EncoderBlock(nn.Module):
    def __init__(self, dim, n_heads, mlp_ratio=2.0):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn = MultiHeadSelfAttention(dim, n_heads)
        self.norm2 = nn.LayerNorm(dim)
        hidden = int(dim * mlp_ratio)
        self.mlp = nn.Sequential(nn.Linear(dim, hidden), nn.GELU(), nn.Linear(hidden, dim))

    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        x = x + self.mlp(self.norm2(x))
        return x

class VisionTransformer(nn.Module):
    def __init__(self, img_size=32, patch=4, n_classes=10, dim=192, depth=6, n_heads=6):
        super().__init__()
        self.patch_embed = PatchEmbedding(img_size, patch, 3, dim)
        n = self.patch_embed.n_patches
        self.cls_token = nn.Parameter(torch.zeros(1, 1, dim))
        self.pos_embed = nn.Parameter(torch.zeros(1, n + 1, dim))
        self.blocks = nn.ModuleList([EncoderBlock(dim, n_heads) for _ in range(depth)])
        self.norm = nn.LayerNorm(dim)
        self.head = nn.Linear(dim, n_classes)
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        nn.init.trunc_normal_(self.cls_token, std=0.02)

    def forward(self, x):
        x = self.patch_embed(x)
        cls = self.cls_token.expand(x.shape[0], -1, -1)
        x = torch.cat([cls, x], dim=1) + self.pos_embed
        for blk in self.blocks:
            x = blk(x)
        return self.head(self.norm(x)[:, 0])

vit = VisionTransformer()
print(f'{sum(p.numel() for p in vit.parameters()):,} parameters')

## 2. Train on CIFAR-10

Heavy weight decay and a cosine schedule are standard for ViTs. Reduce `EPOCHS` if you are on CPU.

In [ ]:
EPOCHS = 15
norm = transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616))
tf = transforms.Compose([transforms.ToTensor(), norm])
train = datasets.CIFAR10('data', train=True, download=True, transform=tf)
test = datasets.CIFAR10('data', train=False, download=True, transform=tf)
train_loader = torch.utils.data.DataLoader(train, batch_size=128, shuffle=True)
test_loader = torch.utils.data.DataLoader(test, batch_size=512)

def evaluate(model):
    model.eval(); correct = total = 0
    with torch.no_grad():
        for x, y in test_loader:
            x, y = x.to(device), y.to(device)
            correct += (model(x).argmax(1) == y).sum().item(); total += y.size(0)
    return correct / total

model = VisionTransformer().to(device)
opt = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.05)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)

for epoch in range(EPOCHS):
    model.train()
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        loss = F.cross_entropy(model(x), y)
        opt.zero_grad(); loss.backward(); opt.step()
    sched.step()
    print(f'epoch {epoch+1}: test acc {evaluate(model)*100:.2f}%')

## 3. The CNN baseline

A small VGG-style CNN with ~6x fewer parameters. On CIFAR-10 from scratch it beats the ViT, because its locality and translation-equivariance priors match how images work.

In [ ]:
class SmallCNN(nn.Module):
    def __init__(self, n_classes=10):
        super().__init__()
        def block(cin, cout):
            return nn.Sequential(
                nn.Conv2d(cin, cout, 3, padding=1), nn.BatchNorm2d(cout), nn.ReLU(),
                nn.Conv2d(cout, cout, 3, padding=1), nn.BatchNorm2d(cout), nn.ReLU(),
                nn.MaxPool2d(2))
        self.features = nn.Sequential(block(3, 32), block(32, 64), block(64, 128))
        self.head = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(),
                                  nn.Dropout(0.1), nn.Linear(128, n_classes))

    def forward(self, x):
        return self.head(self.features(x))

cnn = SmallCNN()
print(f'{sum(p.numel() for p in cnn.parameters()):,} parameters')

## 4. What the ViT learned to look at

Read the class-token attention from the last block back onto the 8x8 patch grid.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def cls_attention(model, x):
    model.eval()
    with torch.no_grad():
        t = model.patch_embed(x.unsqueeze(0).to(device))
        cls = model.cls_token.expand(1, -1, -1)
        t = torch.cat([cls, t], dim=1) + model.pos_embed
        for i, blk in enumerate(model.blocks):
            xn = blk.norm1(t)
            qkv = blk.attn.qkv(xn).reshape(1, xn.shape[1], 3, blk.attn.n_heads, blk.attn.head_dim).permute(2, 0, 3, 1, 4)
            q, k = qkv[0], qkv[1]
            attn = ((q @ k.transpose(-2, -1)) * blk.attn.scale).softmax(-1)
            t = t + blk.attn(xn); t = t + blk.mlp(blk.norm2(t))
        return attn.mean(1)[0, 0, 1:].reshape(8, 8).cpu().numpy()

x, _ = test[7]
amap = cls_attention(model, x)
mean = torch.tensor((0.4914, 0.4822, 0.4465)).view(3,1,1)
std = torch.tensor((0.2470, 0.2435, 0.2616)).view(3,1,1)
img = (x*std+mean).clamp(0,1).permute(1,2,0).numpy()
fig, ax = plt.subplots(1, 2, figsize=(7,3.5))
ax[0].imshow(img); ax[0].set_title('input'); ax[0].axis('off')
ax[1].imshow(amap, cmap='magma'); ax[1].set_title('class-token attention'); ax[1].axis('off')
plt.show()

## Exercises

1. **Patch size.** Re-train with `patch=2` (256 tokens) and `patch=8` (16 tokens). How do accuracy and epoch time change? Remember attention cost grows with the *square* of the token count.
2. **Augmentation.** Add `transforms.RandomCrop(32, padding=4)` and `transforms.RandomHorizontalFlip()` to the training transform. How much does it help the ViT versus the CNN?
3. **Positional embeddings.** Plot the cosine similarity of `model.pos_embed` between positions. Do neighbouring patches end up with similar embeddings?
4. **Depth vs width.** Trade depth for width (e.g. `depth=3, dim=384`) at a fixed parameter budget. Which helps more at this scale?
5. **Remove positional embeddings.** Set `pos_embed` to zero and freeze it. How much accuracy is lost when the model can no longer tell where patches are?